In [4]:
pip install sentencepiece

  Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl.metadata (10 kB)
Using cached sentencepiece-0.2.1-cp313-cp313-win_amd64.whl (1.1 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
import json
import numpy as np
import sentencepiece as spm
from datasets import load_dataset, interleave_datasets
from tqdm import tqdm

In [6]:
MEDICAL_PATH = 'casperhansen/pmc-oa-markdown'
FINEWEB_PATH = 'HuggingFaceFW/fineweb-edu'
FW_NAME = 'sample-10BT'

MODEL_PREFIX = 'med_fine_sp'
VOCAB_SIZE = 50257 
LIMIT_PER_SOURCE = 15000

In [7]:
print("Step 1: Streaming and mixing datasets...")
medical_ds = load_dataset(MEDICAL_PATH, split="train", streaming=True)
fineweb_ds = load_dataset(FINEWEB_PATH, name=FW_NAME, split="train", streaming=True)

Step 1: Streaming and mixing datasets...


Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/49 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

c:\Users\vaibh\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vaibh\.cache\huggingface\hub\datasets--HuggingFaceFW--fineweb-edu. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

In [9]:
mixed_ds = interleave_datasets([medical_ds, fineweb_ds], probabilities=[0.5, 0.5], seed=42)

In [10]:
print("Step 2: Creating corpus for SentencePiece training...")
corpus_file = 'mixed_corpus.txt'
with open(corpus_file, 'w', encoding='utf-8') as f:
    for i, example in enumerate(mixed_ds):
        text = example.get("text", "")
        if text:
            f.write(text.replace('\n', ' ') + "\n")
        if i >= (LIMIT_PER_SOURCE * 2): break

Step 2: Creating corpus for SentencePiece training...


In [11]:
print(f"\nStep 3: Training SentencePiece on mixed data...")
spm.SentencePieceTrainer.train(
    input=corpus_file,
    model_prefix=MODEL_PREFIX,
    vocab_size=VOCAB_SIZE,
    model_type='bpe',
    character_coverage=1.0,
    byte_fallback=True
)


Step 3: Training SentencePiece on mixed data...


In [12]:
print("\nStep 4: Final Tokenization to .npy format...")
sp = spm.SentencePieceProcessor(model_file=f"{MODEL_PREFIX}.model")
all_tokens = []


Step 4: Final Tokenization to .npy format...


In [13]:
with open(corpus_file, 'r', encoding='utf-8') as f:
    for line in tqdm(f):
        ids = sp.encode_as_ids(line)
        all_tokens.extend(ids)

30001it [09:50, 50.82it/s] 


In [14]:
token_array = np.array(all_tokens, dtype=np.uint16)
np.save("mixed_train.npy", token_array)

print(f"\nSuccess! Total tokens in training file: {len(token_array)}")
print(f"Files created: {MODEL_PREFIX}.model, {MODEL_PREFIX}.vocab, mixed_train.npy")


Success! Total tokens in training file: 198714863
Files created: med_fine_sp.model, med_fine_sp.vocab, mixed_train.npy


In [19]:
import sentencepiece as spm

# 1. Load your trained model
sp = spm.SentencePieceProcessor()
sp.load('med_fine_sp.model')

# 2. ENCODE: Text -> IDs (for Model Input)
text = "hospital!! !"
token_ids = sp.encode_as_ids(text)
token_pieces = sp.encode_as_pieces(text)

print(f"Original Text: {text}")
print(f"Token Pieces: {token_pieces}")
print(f"Token IDs:     {token_ids}")

# 3. DECODE: IDs -> Text (for Model Output/Chat)
# This is what you'll use to read what your GPT-2 "writes"
decoded_text = sp.decode_ids(token_ids)
print(f"Decoded Text:  {decoded_text}")

# 4. Check Vocabulary Size
print(f"Vocab Size:    {sp.get_piece_size()}")

Original Text: hospital!! !
Token Pieces: ['▁hospital', '!!', '▁!']
Token IDs:     [5076, 9277, 22201]
Decoded Text:  hospital!! !
Vocab Size:    50257
